# Imports

In [ ]:
import os

import tempfile

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Definindo TOKENIZERS_PARALLELISM=false para o tokenizer não usar threads extras
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
load_dotenv()

In [ ]:
pdf_file_path = './pdf/demonstrativo_financeiro.pdf'

# Explicação

Busca Semântica

- Cada chunk de texto é convertido em um embedding (vetor numérico).
- Esses embeddings são armazenados em um banco vetorial (ex: Chroma).
- Quando o usuário faz uma pergunta (query), ela também é transformada em um embedding.
- O sistema calcula a similaridade vetorial entre a query e todos os embeddings dos chunks.
- Os chunks mais similares são recuperados.
- Esses chunks (texto) são enviados como contexto para o modelo de linguagem (LLM).
- O LLM então gera a resposta com base nesse contexto.

A pergunta é: quais são os vetores que tem o texto mais parecido com o texto da busca (question).\
O modelo retorna os k mais parecidos e entrega o contexto para o LLM.\
Isso é RAG.

# Orquestração do Modelo

## Modelo de Embeddings

https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

This is a sentence-transformers model: It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for semantic search.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True},
)

## Modelo de LLM

In [ ]:
from langchain_groq import ChatGroq

llm_model = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

## Banco Vetorial

In [ ]:
def cria_banco_vetorial(pdf_file, embedding_model):
    if pdf_file is not None:
        try:
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_file:
                with open(pdf_file, 'rb') as f:
                    file = f.read()
                temp_file.write(file)
                file_path = temp_file.name
                
            # Load PDF
            pdf = PyPDFLoader(file_path).load()
            
            text_splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap = 50)
            
            chunks = text_splitter.split_documents(pdf)
            
            print(f"{len(chunks)} chunks criados com sucesso!")
            
            CHROMA_PERSIST_DIR = "chroma_db_persist"

            CHROMA_COLLECTION_NAME = "demonstrativos_financeiros"
            
            vector_store = Chroma.from_documents(
                documents = chunks,
                embedding = embedding_model,
                collection_name = CHROMA_COLLECTION_NAME,
                persist_directory = CHROMA_PERSIST_DIR
            )
        
            os.remove(file_path)
        
        except Exception as e:
            print(f"Erro ao processar PDF: {e}")
    
        return chunks, vector_store
    
    else:
        print("Carregue o arquivo PDF.")

In [ ]:
chunks, vector_store = cria_banco_vetorial(pdf_file_path, embedding_model)

In [ ]:
len(chunks)

In [ ]:
# Cada chunk é um objeto langchain_core.documents.base.Document
type(chunks[0])

In [ ]:
for i, doc in enumerate(chunks):
    print(f"Chunk {i}:")
    print(doc.page_content)
    print("-" * 40)

## Formatação

In [ ]:
def formata_docs(docs):

    return "\n\n---\n\n".join([d.page_content for d in docs])

In [ ]:
formata_docs(chunks)

## Template do Prompt

In [ ]:
RAG_PROMPT_TEMPLATE = """
Você é um assistente de IA especializado em análise financeira.
Sua tarefa é responder perguntas sobre demonstrativos financeiros usando APENAS o contexto fornecido.
Seja direto, preciso e baseie-se exclusivamente nos dados dos trechos.
Se a informação não estiver no contexto, diga "A informação não foi encontrada no documento."

Contexto:
{context}

Pergunta:
{question}

Resposta (em Português):
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

# Chain

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
rag_chain = (
    {"context": retriever | formata_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm_model
    | StrOutputParser()
)

In [ ]:
type(rag_chain)

# Query

In [ ]:
# Valor total das Deduções e Impostos

In [ ]:
question = input("Faça sua pergunta")

In [ ]:
answer = rag_chain.invoke(question)

In [ ]:
answer

# Fins Didáticos

## Visualizar os embeddings

Uma Collection é um container que armazena:

IDs\
embeddings (vetores numéricos)\
documents (texto original)\
metadados (opcional)

In [ ]:
vector_store._collection

In [ ]:
vector_store._collection.count()

In [ ]:
data = vector_store._collection.get(include=["embeddings", "documents"])

In [ ]:
len(data['ids'])

In [ ]:
len(data['embeddings'])

In [ ]:
len(data['documents'])

In [ ]:
data["documents"][0]

In [ ]:
embeddings = data["embeddings"]

In [ ]:
# 13 vetores embeddings com 384 colunas cada.
embeddings

In [ ]:
print(embeddings[0][:10])

In [ ]:
len(embeddings[0])

In [ ]:
retrieved_docs = retriever.invoke(question)
retrieved_docs

In [ ]:
chunks[3].page_content

In [ ]:
retrieved_docs[0].page_content